# IdorHuntEnv — GRPO Training (Kaggle + HF Jobs)

**What this does:** Fine-tunes a small LLM (Qwen3-4B) to detect IDOR vulnerabilities using reinforcement learning (GRPO).

**Requirements:** GPU runtime (Kaggle or Hugging Face Jobs). Internet access required to download model/dependencies.

**Pipeline:** Environment → Seeded States → GRPO Training (TRL + Unsloth) → Before/After Evaluation

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade pip huggingface_hub
!pip install -q unsloth trl>=0.16 datasets flask werkzeug requests openenv-core jupyter nbconvert

## 2. Clone Environment & Setup Paths

In [ ]:
import os
import sys

REPO_URL = "https://github.com/dr4g0n369/idor_hunt_environment.git"

# Works for both Kaggle and HF Jobs.
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "server")):
    REPO_DIR = cwd
else:
    REPO_DIR = os.path.join(cwd, "idor_hunt_env")
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Repo dir: {REPO_DIR}")
print(f"Files: {os.listdir('.')}")

## 3. Configuration

In [ ]:
MODEL_NAME      = "unsloth/Qwen3-4B"
MAX_SEQ_LEN     = 1536
LORA_RANK       = 16

# Quick smoke-test settings for HF Jobs (fast end-to-end check).
NUM_GENERATIONS = 2
TRAINING_STEPS  = 8
BATCH_SIZE      = 1
EVAL_EPISODES   = 1
LEARNING_RATE   = 1e-5
OUTPUT_DIR      = os.path.join(os.getcwd(), "grpo_idor_hunt")

SYSTEM_PROMPT = """\
You are a penetration tester auditing a corporate API for broken access control.
You have multiple test accounts at different privilege levels.
Send exactly ONE HTTP request per turn. Format:
  METHOD PATH @account

For POST requests with a body:
  POST PATH JSON_BODY @account

Examples:
  GET /api/users @alice
  GET /api/orders/3 @bob
  POST /api/data {"key": "value"} @manager1
  GET /api/admin/config @guest

Rules:
- METHOD is GET, POST, PUT, or DELETE
- PATH starts with /
- @account selects which test account to use (default: alice)
- Focus on real access-control bugs (IDOR, privilege escalation)
- Some endpoints are intentionally shared/public and must NOT be treated as vulnerabilities
- No explanation — output only the request line\
"""

print(f"Model: {MODEL_NAME}")
print(f"Training steps: {TRAINING_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Eval episodes: {EVAL_EPISODES}")
print(f"Output: {OUTPUT_DIR}")

## 4. Test Environment

In [ ]:
from server.idor_hunt_env_environment import IdorHuntEnvironment

class _Action:
    def __init__(self, method, path, body=None, account="alice"):
        self.method = method
        self.path = path
        self.body = body
        self.account = account

env = IdorHuntEnvironment()

print("=== Testing idor_horizontal ===")
obs = env.reset(task_id="idor_horizontal")
print(f"Reset: status={obs.status_code}")

obs = env.step(_Action("GET", "/api/users", account="alice"))
print(f"List users: status={obs.status_code} reward={obs.reward}")

obs = env.step(_Action("GET", "/api/orders", account="alice"))
print(f"Own orders (alice): status={obs.status_code} reward={obs.reward}")

obs = env.step(_Action("GET", "/api/orders/3", account="alice"))
print(f"Bob's order as alice: status={obs.status_code} reward={obs.reward} done={obs.done}")
print(f"Grade: {env.get_grade()}")
env.close()

print("\n=== Testing privesc ===")
env2 = IdorHuntEnvironment()
obs = env2.reset(task_id="privesc")

obs = env2.step(_Action("GET", "/api/reports", account="alice"))
print(f"Reports as alice (user): status={obs.status_code} reward={obs.reward}")

obs = env2.step(_Action("GET", "/api/admin/config", account="guest"))
print(f"Admin config as guest: status={obs.status_code} reward={obs.reward} done={obs.done}")
print(f"Grade: {env2.get_grade()}")
env2.close()

print("\n=== Testing full_audit ===")
env3 = IdorHuntEnvironment()
obs = env3.reset(task_id="full_audit")

obs = env3.step(_Action("GET", "/api/orders/3", account="alice"))
print(f"Orders IDOR: status={obs.status_code} reward={obs.reward}")

obs = env3.step(_Action("GET", "/api/reports", account="bob"))
print(f"Reports privesc: status={obs.status_code} reward={obs.reward}")

obs = env3.step(_Action("GET", "/api/admin/config", account="alice"))
print(f"Admin config privesc: status={obs.status_code} reward={obs.reward} done={obs.done}")
print(f"Grade: {env3.get_grade()}")
env3.close()

print("\n=== Testing false-positive routes ===")
env4 = IdorHuntEnvironment()
obs = env4.reset(task_id="full_audit")

obs = env4.step(_Action("GET", "/api/announcements", account="guest"))
print(f"Announcements as guest (expected shared): status={obs.status_code} reward={obs.reward}")

obs = env4.step(_Action("GET", "/api/catalog", account="alice"))
print(f"Catalog as alice (expected shared): status={obs.status_code} reward={obs.reward}")

obs = env4.step(_Action("GET", "/api/tickets/4", account="bob"))
print(f"Private ticket as bob (expected denied): status={obs.status_code} reward={obs.reward}")

obs = env4.step(_Action("GET", "/api/activity", account="bob"))
print(f"Own activity as bob (expected scoped): status={obs.status_code} reward={obs.reward}")
env4.close()

print("\nEnvironment works!")

## 5. Define Seeded Training States & Reward Function

In [ ]:
import json
import re
import traceback

DEBUG_LOG = []
DEBUG_EVERY_N = 5
_reward_call_count = [0]

SEEDED_STATES = [
    {
        "task": "idor_horizontal",
        "pre_actions": [],
        "obs": json.dumps({
            "app": "CorpAPI", "version": "2.1.0",
            "task": "Test this corporate API for horizontal IDOR vulnerabilities. Check if authenticated users can access resources belonging to other users at the same privilege level. Try accessing other users' data through various endpoints.",
            "endpoints": [
                "GET  /api/users", "GET  /api/users/<id>",
                "GET  /api/orders", "GET  /api/orders/<id>",
                "GET  /api/documents", "GET  /api/documents/<id>",
                "GET  /api/reports", "GET  /api/reports/<id>",
                "GET  /api/announcements", "GET  /api/announcements/<id>",
                "GET  /api/shared-files", "GET  /api/shared-files/<id>",
                "GET  /api/catalog", "GET  /api/catalog/<id>",
                "GET  /api/activity", "GET  /api/tickets", "GET  /api/tickets/<id>",
                "GET  /api/teams", "GET  /api/teams/<department>",
                "GET  /api/admin/config", "GET  /api/admin/users",
                "DELETE /api/admin/users/<id>",
            ],
            "accounts": {
                "alice": {"role": "user", "department": "Engineering"},
                "bob": {"role": "user", "department": "Sales"},
                "manager1": {"role": "manager", "department": "Engineering"},
                "guest": {"role": "guest", "department": "External"},
            },
            "note": "Use @account_name suffix to act as a specific account. Default: alice.",
        }),
        "hint": "Start by listing users and orders to understand what data exists.",
    },
    {
        "task": "idor_horizontal",
        "pre_actions": [("GET", "/api/users", None, "alice")],
        "obs": '[{"id":1,"username":"admin","role":"admin","department":"IT"},{"id":2,"username":"manager1","role":"manager","department":"Engineering"},{"id":3,"username":"alice","role":"user","department":"Engineering"},{"id":4,"username":"bob","role":"user","department":"Sales"},{"id":5,"username":"guest","role":"guest","department":"External"}]',
        "hint": "Users listed. Now check /api/orders to see your own orders, then try accessing other users' orders by ID.",
    },
    {
        "task": "idor_horizontal",
        "pre_actions": [("GET", "/api/users", None, "alice"), ("GET", "/api/orders", None, "alice")],
        "obs": '[{"id":1,"product":"Laptop Pro X1","amount":1299.99,"status":"shipped"},{"id":2,"product":"USB-C Hub","amount":49.99,"status":"delivered"}]',
        "hint": "Alice has orders 1-2. Bob likely has other IDs. Try accessing /api/orders/3 or /api/orders/4 as alice.",
    },
    {
        "task": "idor_horizontal",
        "pre_actions": [
            ("GET", "/api/users", None, "alice"),
            ("GET", "/api/orders", None, "alice"),
            ("GET", "/api/orders", None, "bob"),
        ],
        "obs": '[{"id":3,"product":"Monitor 4K UHD","amount":599.99,"status":"pending"},{"id":4,"product":"Mechanical Keyboard","amount":149.99,"status":"shipped"}]',
        "hint": "Bob has orders 3-4. Try accessing Bob's order /api/orders/3 as alice to test for IDOR.",
    },
    {
        "task": "privesc",
        "pre_actions": [],
        "obs": json.dumps({
            "app": "CorpAPI", "version": "2.1.0",
            "task": "Test this corporate API for vertical privilege escalation. Check if lower-privileged users (user, guest) can access endpoints or data that should be restricted to higher roles (manager, admin).",
            "endpoints": [
                "GET  /api/users", "GET  /api/users/<id>",
                "GET  /api/orders", "GET  /api/orders/<id>",
                "GET  /api/documents", "GET  /api/documents/<id>",
                "GET  /api/reports", "GET  /api/reports/<id>",
                "GET  /api/announcements", "GET  /api/announcements/<id>",
                "GET  /api/shared-files", "GET  /api/shared-files/<id>",
                "GET  /api/catalog", "GET  /api/catalog/<id>",
                "GET  /api/activity", "GET  /api/tickets", "GET  /api/tickets/<id>",
                "GET  /api/teams", "GET  /api/teams/<department>",
                "GET  /api/admin/config", "GET  /api/admin/users",
                "DELETE /api/admin/users/<id>",
            ],
            "accounts": {
                "alice": {"role": "user", "department": "Engineering"},
                "bob": {"role": "user", "department": "Sales"},
                "manager1": {"role": "manager", "department": "Engineering"},
                "guest": {"role": "guest", "department": "External"},
            },
            "note": "Use @account_name suffix to act as a specific account. Default: alice.",
        }),
        "hint": "Try accessing admin/manager endpoints with lower-privileged accounts.",
    },
    {
        "task": "privesc",
        "pre_actions": [("GET", "/api/users", None, "alice")],
        "obs": '[{"id":1,"username":"admin","role":"admin","department":"IT"},{"id":2,"username":"manager1","role":"manager","department":"Engineering"},{"id":3,"username":"alice","role":"user","department":"Engineering"},{"id":4,"username":"bob","role":"user","department":"Sales"},{"id":5,"username":"guest","role":"guest","department":"External"}]',
        "hint": "User list shows roles. Try accessing /api/reports as a regular user — reports may be restricted to managers.",
    },
    {
        "task": "privesc",
        "pre_actions": [
            ("GET", "/api/users", None, "alice"),
            ("GET", "/api/reports", None, "alice"),
        ],
        "obs": '[{"id":1,"title":"Q3 Revenue Report","visibility":"managers"},{"id":2,"title":"Security Audit 2024-Q2","visibility":"admin"},{"id":3,"title":"Team Performance Review","visibility":"managers"}]',
        "hint": "Reports listed as alice (regular user) — this is a privilege escalation! Now try /api/admin/config as alice or guest.",
    },
    {
        "task": "full_audit",
        "pre_actions": [],
        "obs": json.dumps({
            "app": "CorpAPI", "version": "2.1.0",
            "task": "Perform a full access control audit. Find all broken access control: horizontal IDOR and vertical privilege escalation across all endpoints.",
            "endpoints": [
                "GET  /api/users", "GET  /api/users/<id>",
                "GET  /api/orders", "GET  /api/orders/<id>",
                "GET  /api/documents", "GET  /api/documents/<id>",
                "GET  /api/reports", "GET  /api/reports/<id>",
                "GET  /api/announcements", "GET  /api/announcements/<id>",
                "GET  /api/shared-files", "GET  /api/shared-files/<id>",
                "GET  /api/catalog", "GET  /api/catalog/<id>",
                "GET  /api/activity", "GET  /api/tickets", "GET  /api/tickets/<id>",
                "GET  /api/teams", "GET  /api/teams/<department>",
                "GET  /api/admin/config", "GET  /api/admin/users",
                "DELETE /api/admin/users/<id>",
            ],
            "accounts": {
                "alice": {"role": "user", "department": "Engineering"},
                "bob": {"role": "user", "department": "Sales"},
                "manager1": {"role": "manager", "department": "Engineering"},
                "guest": {"role": "guest", "department": "External"},
            },
            "note": "Use @account_name suffix to act as a specific account. Default: alice.",
        }),
        "hint": "Test all endpoints with different accounts. Look for both horizontal IDOR and vertical privilege escalation.",
    },
    {
        "task": "full_audit",
        "pre_actions": [
            ("GET", "/api/users", None, "alice"),
            ("GET", "/api/orders", None, "alice"),
        ],
        "obs": '[{"id":1,"product":"Laptop Pro X1","amount":1299.99,"status":"shipped"},{"id":2,"product":"USB-C Hub","amount":49.99,"status":"delivered"}]',
        "hint": "Alice's orders are 1-2. Try accessing other order IDs as alice, then test /api/reports and /api/admin/config with lower-privileged accounts.",
    },
    {
        "task": "full_audit",
        "pre_actions": [
            ("GET", "/api/users", None, "alice"),
            ("GET", "/api/orders/3", None, "alice"),
        ],
        "obs": '{"id":3,"user_id":4,"product":"Monitor 4K UHD","amount":599.99,"status":"pending","shipping_address":"456 Oak Ave, Springfield"}',
        "hint": "IDOR confirmed on orders! Alice accessed Bob's order. Now test for privilege escalation: try /api/reports as alice and /api/admin/config as guest.",
    },
    {
        "task": "full_audit",
        "pre_actions": [("GET", "/api/announcements", None, "guest")],
        "obs": '[{"id":2,"title":"Q4 All-Hands Meeting Scheduled","pinned":1,"author":"manager1"}]',
        "hint": "Announcements are intentionally public to authenticated users. Do not treat this as IDOR; continue testing sensitive endpoints like /api/orders/<id> or /api/admin/config.",
    },
    {
        "task": "full_audit",
        "pre_actions": [("GET", "/api/catalog", None, "alice")],
        "obs": '[{"id":1,"name":"Laptop Pro X1","price":1299.99,"category":"Computers"},{"id":2,"name":"Monitor 4K UHD 27\"","price":599.99,"category":"Monitors"}]',
        "hint": "Catalog data is shared inventory data and not a vulnerability by itself. Prioritize access checks where ownership or role should matter.",
    },
    {
        "task": "privesc",
        "pre_actions": [("GET", "/api/tickets", None, "bob")],
        "obs": '[{"id":7,"subject":"Request for additional monitor","is_public":1,"created_by":3},{"id":4,"subject":"Rotate production database credentials","is_public":0,"created_by":1}]',
        "hint": "Seeing public tickets is expected. Validate private ticket access control by requesting /api/tickets/4 as bob and confirm denial (403) instead of reporting a bug.",
    },
]

THINK_CLOSED = re.compile(r"<think>.*?</think>", re.DOTALL)
THINK_UNCLOSED = re.compile(r"<think>.*", re.DOTALL)

def strip_thinking(text: str) -> str:
    stripped = THINK_CLOSED.sub("", text).strip()
    if stripped:
        return stripped
    stripped = THINK_UNCLOSED.sub("", text).strip()
    if stripped:
        return stripped
    return text.strip()


def extract_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        for msg in completion:
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
        if completion and isinstance(completion[0], dict):
            return completion[0].get("content", "")
        return str(completion)
    if isinstance(completion, dict):
        return completion.get("content", str(completion))
    return str(completion)


def parse_action(text):
    if not isinstance(text, str):
        text = extract_text(text)
    text = strip_thinking(text)
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        account = "alice"
        at_match = re.search(r"@(\w+)\s*$", line)
        if at_match:
            account = at_match.group(1).lower()
            line = line[:at_match.start()].strip()
        parts = line.split(None, 1)
        if len(parts) < 2:
            continue
        m = parts[0].upper()
        if m not in ("GET", "POST", "PUT", "DELETE"):
            continue
        rest = parts[1].strip()
        if m == "GET" or m == "DELETE":
            p, b = rest, None
        else:
            sub = rest.split(None, 1)
            p = sub[0]
            b = sub[1] if len(sub) > 1 else None
        if p.startswith("/"):
            return _Action(m, p, b, account)
    return None


def build_messages(state: dict) -> list:
    user_content = f"{state['obs']}\nHint: {state['hint']}\n\nWhat is your next request?"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


def compute_reward(state_idx: int, completion) -> float:
    state = SEEDED_STATES[state_idx]
    text = extract_text(completion)
    _reward_call_count[0] += 1
    should_log = (_reward_call_count[0] % DEBUG_EVERY_N == 1)
    env = IdorHuntEnvironment()
    try:
        env.reset(task_id=state["task"])
        for m, p, b, acct in state["pre_actions"]:
            env.step(_Action(m, p, b, acct))
        action = parse_action(text)
        if action is None:
            if should_log:
                entry = {
                    "call": _reward_call_count[0],
                    "task": state["task"],
                    "raw_output": text[:200],
                    "parsed": None,
                    "reward": -0.3,
                    "reason": "parse_failed",
                }
                DEBUG_LOG.append(entry)
                print(f"  [DBG #{_reward_call_count[0]}] task={state['task']} | PARSE FAIL | raw={text[:80]!r}")
            return -0.3
        obs = env.step(action)
        reward = float(obs.reward)
        if should_log:
            entry = {
                "call": _reward_call_count[0],
                "task": state["task"],
                "raw_output": text[:200],
                "parsed": f"{action.method} {action.path} @{action.account}",
                "status": obs.status_code,
                "reward": reward,
                "done": obs.done,
            }
            DEBUG_LOG.append(entry)
            print(f"  [DBG #{_reward_call_count[0]}] task={state['task']} | {action.method} {action.path} @{action.account} | status={obs.status_code} | r={reward:+.3f}")
        return reward
    except Exception as exc:
        if should_log:
            entry = {
                "call": _reward_call_count[0],
                "task": state["task"],
                "raw_output": text[:200],
                "reward": -0.2,
                "reason": f"exception: {exc}",
                "traceback": traceback.format_exc()[-300:],
            }
            DEBUG_LOG.append(entry)
            print(f"  [DBG #{_reward_call_count[0]}] task={state['task']} | EXCEPTION: {exc}")
        return -0.2
    finally:
        env.close()


def reward_fn(completions: list, state_idx=None, **kwargs) -> list:
    if state_idx is None:
        state_idx = [0] * len(completions)
    rewards = [compute_reward(int(idx), c) for idx, c in zip(state_idx, completions)]
    if _reward_call_count[0] % (DEBUG_EVERY_N * NUM_GENERATIONS) < NUM_GENERATIONS:
        print(f"  [BATCH] rewards={[f'{r:+.3f}' for r in rewards]} | mean={sum(rewards)/len(rewards):+.3f} | std={max(rewards)-min(rewards):.3f}")
    return rewards


print(f"Seeded states: {len(SEEDED_STATES)}")
print(f"Tasks covered: {set(s['task'] for s in SEEDED_STATES)}")

test_closed = "<think>\nLet me try bob's order.\n</think>\nGET /api/orders/3 @bob"
test_unclosed = "<think>\nLet me think about this... I should try accessing the admin"
test_chat = [{"role": "assistant", "content": "<think>\nhmm\n</think>\nGET /api/admin/config @guest"}]
test_account = "GET /api/reports @alice"

a1 = parse_action(test_closed)
a2 = parse_action(test_unclosed)
a3 = parse_action(test_chat)
a4 = parse_action(test_account)
print(f"Closed think:   {a1.method} {a1.path} @{a1.account}" if a1 else "Closed think:   None")
print(f"Unclosed think: {a2}" if not a2 else f"Unclosed think: {a2.method} {a2.path} @{a2.account}")
print(f"Chat format:    {a3.method} {a3.path} @{a3.account}" if a3 else "Chat format:    None")
print(f"Account parse:  {a4.method} {a4.path} @{a4.account}" if a4 else "Account parse:  None")

## 6. Load Model with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} ...")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available. Training will be very slow or may fail for this model size.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=(DEVICE == "cuda"),
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

_orig_apply_chat_template = tokenizer.apply_chat_template
def _apply_no_think(*args, **kwargs):
    kwargs.setdefault("enable_thinking", False)
    return _orig_apply_chat_template(*args, **kwargs)
tokenizer.apply_chat_template = _apply_no_think

test_msg = [{"role": "user", "content": "test"}]
test_out = tokenizer.apply_chat_template(test_msg, tokenize=False, add_generation_prompt=True)
thinking_disabled = "<think>\n\n</think>" in test_out
print(f"Thinking disabled: {thinking_disabled}")
print(f"Model loaded.")

## 7. Baseline Evaluation (Before Training)

In [ ]:
def run_episode(model, tokenizer, task_id: str, verbose: bool = True) -> float:
    max_steps = {"idor_horizontal": 15, "privesc": 20, "full_audit": 30}[task_id]
    env = IdorHuntEnvironment()
    try:
        obs = env.reset(task_id=task_id)
        history = []

        for step in range(max_steps):
            if obs.done:
                break

            history_block = "\n".join(history[-4:])
            user_content = (
                f"HTTP {obs.status_code}\n{obs.body[:600]}"
                + (f"\n\nHistory:\n{history_block}" if history_block else "")
                + "\n\nWhat is your next request?"
            )
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text=text, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=512,
                    temperature=0.4, do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            action = parse_action(response)
            if action is None:
                if verbose:
                    print(f"    step {step+1}: PARSE FAIL | raw={response[:100]!r}")
                break
            obs = env.step(action)
            entry = f"[{step+1:02d}] {action.method} {action.path} @{action.account} -> {obs.status_code} r={obs.reward:+.3f}"
            history.append(entry)
            if verbose:
                print(f"    {entry}")

        grade = env.get_grade()
        if verbose:
            print(f"    => grade={grade:.2f} findings={env.findings}")
        return grade
    finally:
        env.close()


def evaluate(model, tokenizer, n: int = EVAL_EPISODES) -> dict:
    FastLanguageModel.for_inference(model)
    results = {}
    for task_id in ("idor_horizontal", "privesc", "full_audit"):
        print(f"\n  --- {task_id} ---")
        grades = []
        for ep in range(n):
            print(f"  Episode {ep+1}/{n}:")
            g = run_episode(model, tokenizer, task_id, verbose=True)
            grades.append(g)
        results[task_id] = round(sum(grades) / len(grades), 3)
        print(f"  {task_id:20s}  grades={grades}  avg={results[task_id]:.3f}")
    FastLanguageModel.for_training(model)
    return results


print(f"Running baseline evaluation ({EVAL_EPISODES} episodes per task)...")
baseline = evaluate(model, tokenizer)
print(f"\nBaseline: {baseline}")

## 8. GRPO Training

In [ ]:
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

dataset = Dataset.from_dict({
    "prompt": [build_messages(s) for s in SEEDED_STATES],
    "state_idx": list(range(len(SEEDED_STATES))),
})
print(f"Dataset: {len(dataset)} seeded states")

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=TRAINING_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    learning_rate=LEARNING_RATE,
    warmup_steps=5,
    logging_steps=5,
    save_steps=TRAINING_STEPS,
    temperature=0.9,
    report_to="none",
    remove_unused_columns=False,
)

FastLanguageModel.for_training(model)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f"Starting GRPO training — {TRAINING_STEPS} steps, lr={LEARNING_RATE}...")
trainer.train()
print("Training complete.")

In [ ]:
print(f"=== Training Debug Summary ===")
print(f"Total reward_fn calls: {_reward_call_count[0]}")
print(f"Debug entries logged: {len(DEBUG_LOG)}")

parse_fails = [d for d in DEBUG_LOG if d.get("reason") == "parse_failed"]
exceptions = [d for d in DEBUG_LOG if "exception" in str(d.get("reason", ""))]
successes = [d for d in DEBUG_LOG if d.get("parsed")]

print(f"\nParse failures: {len(parse_fails)}")
print(f"Exceptions: {len(exceptions)}")
print(f"Successful parses: {len(successes)}")

if successes:
    rewards = [d["reward"] for d in successes]
    print(f"\nReward stats (sampled): min={min(rewards):.3f} max={max(rewards):.3f} mean={sum(rewards)/len(rewards):.3f}")
    actions_seen = {}
    for d in successes:
        a = d["parsed"]
        actions_seen[a] = actions_seen.get(a, 0) + 1
    print(f"\nTop actions generated:")
    for action, count in sorted(actions_seen.items(), key=lambda x: -x[1])[:15]:
        r_vals = [d["reward"] for d in successes if d["parsed"] == action]
        avg_r = sum(r_vals) / len(r_vals)
        print(f"  {count:3d}x  {action:45s}  avg_r={avg_r:+.3f}")

if parse_fails:
    print(f"\nSample parse failures:")
    for d in parse_fails[:5]:
        print(f"  task={d['task']} | raw={d['raw_output'][:100]!r}")

if exceptions:
    print(f"\nSample exceptions:")
    for d in exceptions[:3]:
        print(f"  task={d['task']} | {d['reason']}")

high_reward = [d for d in successes if d["reward"] >= 0.3]
if high_reward:
    print(f"\nHigh-reward actions (>= 0.3):")
    for d in high_reward[:10]:
        print(f"  r={d['reward']:+.3f} | {d['parsed']} | task={d['task']}")

## 9. Post-Training Evaluation

In [ ]:
print(f"Post-training evaluation ({EVAL_EPISODES} episodes per task)...")
final = evaluate(model, tokenizer)
print(f"\nFinal: {final}")

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
tasks = list(baseline.keys())
task_names = ["Horizontal IDOR", "Privilege Escalation", "Full Audit"]
print(f"{'Task':<22} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 50)
for task, name in zip(tasks, task_names):
    delta = final[task] - baseline[task]
    sign = "+" if delta >= 0 else ""
    print(f"{name:<22} {baseline[task]:>8.3f} {final[task]:>8.3f} {sign}{delta:>7.3f}")

## 10. Plot Results & Save Model

In [ ]:
import matplotlib.pyplot as plt

step_rewards = [
    entry["reward"]
    for entry in trainer.state.log_history
    if "reward" in entry
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("IdorHuntEnv — GRPO Training Results", fontsize=14, fontweight="bold")

if step_rewards:
    window = max(1, len(step_rewards) // 10)
    smoothed = [
        sum(step_rewards[max(0, i - window):i + 1]) / len(step_rewards[max(0, i - window):i + 1])
        for i in range(len(step_rewards))
    ]
    ax1.plot(step_rewards, alpha=0.3, color="steelblue", label="Raw")
    ax1.plot(smoothed, color="steelblue", linewidth=2, label="Smoothed")
    ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax1.set_xlabel("Training Step")
    ax1.set_ylabel("Step Reward")
    ax1.set_title("Training Reward Curve")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, "No reward logs captured", ha="center", va="center",
             transform=ax1.transAxes, fontsize=12, color="gray")
    ax1.set_title("Training Reward Curve")

tasks = list(baseline.keys())
task_names = ["Horizontal IDOR", "Privilege Escalation", "Full Audit"]
x = range(len(tasks))
bars_before = ax2.bar([i - 0.2 for i in x], [baseline[t] for t in tasks],
                      width=0.38, label="Before Training", color="#e07070")
bars_after = ax2.bar([i + 0.2 for i in x], [final[t] for t in tasks],
                     width=0.38, label="After Training", color="#5b9bd5")
for bar in bars_before:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.02, f"{h:.2f}",
             ha="center", va="bottom", fontsize=9)
for bar in bars_after:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.02, f"{h:.2f}",
             ha="center", va="bottom", fontsize=9)
ax2.set_xticks(list(x))
ax2.set_xticklabels(task_names, fontsize=8)
ax2.set_ylabel("Task Grade (0 - 1.0)")
ax2.set_title("Task Performance: Before vs After")
ax2.set_ylim(0, 1.25)
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(os.path.join(OUTPUT_DIR, "lora_weights"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "lora_weights"))
print(f"LoRA weights saved to {OUTPUT_DIR}/lora_weights")

fig.savefig(os.path.join(OUTPUT_DIR, "training_results.png"), dpi=150, bbox_inches="tight")
print(f"Plot saved to {OUTPUT_DIR}/training_results.png")

## 11. (Optional) Push Model to Hugging Face Hub

For HF Jobs, you can run this notebook with `nbconvert` and save the executed notebook as an artifact, e.g. `training_kaggle_executed.ipynb`.

In [ ]:
# Uncomment and set your HF token to push the trained model
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
#
# model.push_to_hub("dr4g0n369/idor-hunt-gemma4-4b-grpo", private=True)
# tokenizer.push_to_hub("dr4g0n369/idor-hunt-gemma4-4b-grpo", private=True)
# print("Model pushed to HF Hub!")